# Multi-Crop Leaf Disease Detection - Kaggle Training + Quantization (Kaggle Dataset)

**This notebook trains MobileNetV2 and EfficientNet-Lite0 models and exports TFLite (dynamic + full INT8) using a Kaggle Dataset for input and Kaggle Working storage for outputs.**

**Prerequisites:**
- Notebook Settings -> Accelerator -> GPU
- Add your dataset in the right panel (Add data)
- Dataset folder contains `processed/train`, `processed/val`, `processed/test`

**Key differences (Kaggle):**
- Dataset is read from `/kaggle/input/<dataset-name>` (read-only)
- Models and TFLite exports are saved to `/kaggle/working`
- Use `Save Version` to export results as a Kaggle Dataset

## Step 1: Set Kaggle dataset path

In [ ]:
import os

print('Available datasets in /kaggle/input:')
print(os.listdir('/kaggle/input'))

# TODO: change to your Kaggle dataset slug
KAGGLE_DATASET = '/kaggle/input/datasets/mdhasibultamim/multi-crop-leaf-disease-dataset'
DATASET_BASE = os.path.join(KAGGLE_DATASET, 'processed')

if os.path.exists(DATASET_BASE):
    print('Found dataset:', DATASET_BASE)
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(DATASET_BASE, split)
        if os.path.exists(split_path):
            n_classes = len([
                d for d in os.listdir(split_path)
                if os.path.isdir(os.path.join(split_path, d))
            ])
            print(f'  OK {split}: {n_classes} classes')
        else:
            print(f'  MISSING {split}')
else:
    print('Dataset not found:', DATASET_BASE)
    print('Update KAGGLE_DATASET to your dataset slug.')

## Step 2: Optional - copy dataset to /kaggle/working (faster IO)

In [ ]:
import os
import shutil
import subprocess

if "DATASET_BASE" not in globals():
    raise ValueError("DATASET_BASE not set. Run Step 1 first.")

source_dataset = DATASET_BASE
working_dataset = "/kaggle/working/leaf_data/processed"

if os.path.exists(source_dataset):
    os.makedirs("/kaggle/working/leaf_data", exist_ok=True)

    if not os.path.exists(working_dataset):
        print("Copying dataset to /kaggle/working (this may take a while)...")
        try:
            subprocess.run(
                ["rsync", "-a", "--info=progress2", f"{source_dataset}/", f"{working_dataset}/"],
                check=True,
            )
        except Exception as e:
            print(f"rsync failed ({e}); falling back to shutil.copytree...")
            shutil.copytree(source_dataset, working_dataset, dirs_exist_ok=True)
    else:
        print("Working dataset already exists:", working_dataset)

    if os.path.exists(os.path.join(working_dataset, "train")):
        dataset_base = working_dataset
    else:
        dataset_base = source_dataset
    print("Using dataset:", dataset_base)
else:
    print("Source dataset not found:", source_dataset)

## Step 3: Clone Project Repository

In [ ]:
# Step 3a: Copy repo from Kaggle Model (read-only) to /kaggle/working
import os
import shutil

repo_src = (
    "/kaggle/input/models/mdhasibultamim/"
    "multi-crop-leaf-disease-detection-system/tensorflow2/default/1/training"
 )
repo_dir = "/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"

if not os.path.exists(repo_src):
    raise FileNotFoundError(f"Repo source not found: {repo_src}")

if not os.path.exists(repo_dir):
    print("Copying repo to /kaggle/working...")
    shutil.copytree(repo_src, repo_dir, dirs_exist_ok=True)
else:
    print("Repo already exists:", repo_dir)

print("Repo ready:", repo_dir)

In [ ]:
import os

repo_dir = "/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
if not os.path.exists(repo_dir):
    raise FileNotFoundError(
        "Repo not found. Run Step 3a to copy it from Kaggle Models."
    )

%cd {repo_dir}
print("Repository ready!")

## Step 4: Install Dependencies

In [ ]:
%pip install -q -r requirements.txt
print('Dependencies installed successfully!')

**Preprocessing note:** Training and quantization use `preprocess_input` for the selected architecture. Make sure your Flutter app uses the same normalization (`mobilenet_v2` or `efficientnet`) when running inference.

## Step 5: Configure Training for MobileNetV2 (2-Phase Schedule)

In [ ]:
import os
import yaml

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
cfg_path = os.path.join(repo_dir, 'training', 'config_mobilenetv2.yaml')

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

output_base = '/kaggle/working/leaf_models'
os.makedirs(output_base, exist_ok=True)

with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['data_dir'] = dataset_base
cfg['dataset']['train_dir'] = '{}/train'.format(dataset_base)
cfg['dataset']['val_dir'] = '{}/val'.format(dataset_base)
cfg['dataset']['test_dir'] = '{}/test'.format(dataset_base)

train_dir = cfg['dataset']['train_dir']
if not os.path.exists(train_dir):
    raise FileNotFoundError('Train directory not found: {}'.format(train_dir))

num_classes = len([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])
cfg['model']['num_classes'] = num_classes

cfg['training']['epochs'] = 25
cfg['training']['freeze_base'] = True
cfg['training']['unfreeze_epoch'] = 5
cfg['training']['freeze_until_layer'] = 50
cfg['training']['fine_tune_learning_rate'] = 0.0001

cfg['optimizer']['learning_rate'] = 0.001
cfg['lr_schedule']['type'] = 'reduce_on_plateau'
cfg['lr_schedule']['monitor'] = 'val_loss'
cfg['lr_schedule']['factor'] = 0.5
cfg['lr_schedule']['patience'] = 5
cfg['lr_schedule']['min_lr'] = 1e-7

cfg['callbacks']['early_stopping']['patience'] = 5

cfg['export']['save_dir'] = output_base

cfg['callbacks']['tensorboard']['enabled'] = False
cfg['callbacks']['csv_logger']['filename'] = '{}/training_log_mobilenetv2.csv'.format(output_base)

with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Configuration updated for Kaggle')
print('Training schedule: 5 epochs frozen + 20 epochs fine-tune')
print('Unfreeze from layer: 50')
print('Classes: {}'.format(cfg['model']['num_classes']))
print('Train dataset: {}'.format(cfg['dataset']['train_dir']))
print('Models save to: {}'.format(cfg['export']['save_dir']))
print('TensorBoard logs: DISABLED (saves space)')

## Step 6: Train MobileNetV2

In [ ]:
%cd /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training

!python train.py --config config_mobilenetv2.yaml

## Step 7: Configure EfficientNet-Lite0 (2-Phase Schedule)

In [ ]:
import os
import yaml

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
cfg_path = os.path.join(repo_dir, 'training', 'config_efficientnet_lite0.yaml')

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

output_base = '/kaggle/working/leaf_models'
os.makedirs(output_base, exist_ok=True)

with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['data_dir'] = dataset_base
cfg['dataset']['train_dir'] = '{}/train'.format(dataset_base)
cfg['dataset']['val_dir'] = '{}/val'.format(dataset_base)
cfg['dataset']['test_dir'] = '{}/test'.format(dataset_base)

train_dir = cfg['dataset']['train_dir']
if not os.path.exists(train_dir):
    raise FileNotFoundError('Train directory not found: {}'.format(train_dir))

num_classes = len([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])
cfg['model']['num_classes'] = num_classes

cfg['training']['epochs'] = 25
cfg['training']['freeze_base'] = True
cfg['training']['unfreeze_epoch'] = 5
cfg['training']['freeze_until_layer'] = 50
cfg['training']['fine_tune_learning_rate'] = 0.0001

cfg['optimizer']['learning_rate'] = 0.001
cfg['lr_schedule']['type'] = 'reduce_on_plateau'
cfg['lr_schedule']['monitor'] = 'val_loss'
cfg['lr_schedule']['factor'] = 0.5
cfg['lr_schedule']['patience'] = 5
cfg['lr_schedule']['min_lr'] = 1e-7

cfg['callbacks']['early_stopping']['patience'] = 5

cfg['export']['save_dir'] = output_base

cfg['callbacks']['tensorboard']['enabled'] = False
cfg['callbacks']['csv_logger']['filename'] = '{}/training_log_efficientnet_lite0.csv'.format(output_base)

with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('EfficientNet-Lite0 config updated for Kaggle')
print('Training schedule: 5 epochs frozen + 20 epochs fine-tune')
print('Unfreeze from layer: 50')
print('Classes: {}'.format(cfg['model']['num_classes']))
print('Models save to: {}'.format(cfg['export']['save_dir']))

## Step 8: Train EfficientNet-Lite0

In [ ]:
%cd /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training

!python train.py --config config_efficientnet_lite0.yaml

## Step 9: List Trained Models and Exports

In [ ]:
import os

models_dir = '/kaggle/working/leaf_models'

print('Trained models saved to /kaggle/working:\n')

for arch in ['mobilenetv2', 'efficientnet_lite0']:
    arch_dir = os.path.join(models_dir, arch)
    if os.path.exists(arch_dir):
        print('{}:'.format(arch))
        for f in sorted(os.listdir(arch_dir)):
            fpath = os.path.join(arch_dir, f)
            if os.path.isfile(fpath):
                fsize = os.path.getsize(fpath) / (1024**2)
                print('  - {} ({:.1f} MB)'.format(f, fsize))
            else:
                print('  - {}/ (dir)'.format(f))
    else:
        print('{}: No models found yet'.format(arch))

## Step 10: Evaluate a Trained Model (MobileNetV2 Example)

In [ ]:
import os
import glob
import subprocess

# Choose architecture to evaluate
arch = 'mobilenetv2'  # or 'efficientnet_lite0'
models_dir = '/kaggle/working/leaf_models/{}'.format(arch)

def find_latest_model(arch_dir):
    saved_models = sorted(
        glob.glob(os.path.join(arch_dir, 'saved_model_*')),
        key=os.path.getmtime,
    )
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(
        glob.glob(os.path.join(arch_dir, '*.h5')),
        key=os.path.getmtime,
    )
    if h5_models:
        return h5_models[-1]
    return None

if os.path.exists(models_dir):
    model_path = find_latest_model(models_dir)
    if model_path:
        print('Evaluating: {}\n'.format(os.path.basename(model_path)))
        training_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training'
        os.chdir(training_dir)
        config_name = (
            'config_mobilenetv2.yaml'
            if arch == 'mobilenetv2'
            else 'config_efficientnet_lite0.yaml'
        )
        subprocess.run(['python', 'evaluate.py', '--model', model_path, '--config', config_name])
    else:
        print('No model found for evaluation')
else:
    print('Directory not found: {}'.format(models_dir))

## Step 11: Quantize Models (Dynamic Range)

In [ ]:
import os
import glob
import subprocess

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
quant_script = os.path.join(repo_dir, 'quantization', 'post_training_quant.py')
models_root = '/kaggle/working/leaf_models'
output_dir = '/kaggle/working/leaf_models/quantized'
os.makedirs(output_dir, exist_ok=True)

def find_latest_model(arch_dir):
    saved_models = sorted(
        glob.glob(os.path.join(arch_dir, 'saved_model_*')),
        key=os.path.getmtime,
    )
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(
        glob.glob(os.path.join(arch_dir, '*.h5')),
        key=os.path.getmtime,
    )
    if h5_models:
        return h5_models[-1]
    return None

for arch in ['mobilenetv2', 'efficientnet_lite0']:
    arch_dir = os.path.join(models_root, arch)
    model_path = find_latest_model(arch_dir)
    if not model_path:
        print('{}: No model found for quantization'.format(arch))
        continue
    out_path = os.path.join(output_dir, '{}_dynamic.tflite'.format(arch))
    print('Quantizing (dynamic range): {}'.format(arch))
    subprocess.run(
        [
            'python',
            quant_script,
            '--model_path',
            model_path,
            '--output_path',
            out_path,
            '--arch',
            arch,
        ],
        check=False,
    )

## Step 12: Quantize Models (Full INT8)

In [ ]:
import os
import glob
import subprocess

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
quant_script = os.path.join(repo_dir, 'quantization', 'post_training_quant.py')
models_root = '/kaggle/working/leaf_models'
output_dir = '/kaggle/working/leaf_models/quantized'

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

rep_dir = '{}/train'.format(dataset_base)
test_dir = '{}/test'.format(dataset_base)
os.makedirs(output_dir, exist_ok=True)

def find_latest_model(arch_dir):
    saved_models = sorted(
        glob.glob(os.path.join(arch_dir, 'saved_model_*')),
        key=os.path.getmtime,
    )
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(
        glob.glob(os.path.join(arch_dir, '*.h5')),
        key=os.path.getmtime,
    )
    if h5_models:
        return h5_models[-1]
    return None

for arch in ['mobilenetv2', 'efficientnet_lite0']:
    arch_dir = os.path.join(models_root, arch)
    model_path = find_latest_model(arch_dir)
    if not model_path:
        print('{}: No model found for INT8 quantization'.format(arch))
        continue
    out_path = os.path.join(output_dir, '{}_int8.tflite'.format(arch))
    print('Quantizing (full INT8): {}'.format(arch))
    subprocess.run(
        [
            'python',
            quant_script,
            '--model_path',
            model_path,
            '--output_path',
            out_path,
            '--representative_data',
            rep_dir,
            '--evaluate',
            '--test_data',
            test_dir,
            '--arch',
            arch,
        ],
        check=False,
    )

## Step 13: Benchmark TFLite Models (Kaggle CPU)

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf

quant_dir = '/kaggle/working/leaf_models/quantized'

if not os.path.exists(quant_dir):
    print('Quantized models directory not found: {}'.format(quant_dir))
else:
    tflite_files = [f for f in os.listdir(quant_dir) if f.endswith('.tflite')]
    if not tflite_files:
        print('No .tflite files found in {}'.format(quant_dir))
    else:
        def benchmark_tflite(path, runs=100):
            interpreter = tf.lite.Interpreter(model_path=path)
            interpreter.allocate_tensors()
            input_details = interpreter.get_input_details()
            input_shape = input_details[0]['shape']
            input_dtype = input_details[0]['dtype']

            if input_dtype == np.uint8:
                dummy = np.random.randint(0, 255, size=input_shape, dtype=np.uint8)
            else:
                dummy = np.random.rand(*input_shape).astype(np.float32)

            for _ in range(10):
                interpreter.set_tensor(input_details[0]['index'], dummy)
                interpreter.invoke()

            times = []
            for _ in range(runs):
                start = time.perf_counter()
                interpreter.set_tensor(input_details[0]['index'], dummy)
                interpreter.invoke()
                times.append((time.perf_counter() - start) * 1000)

            avg = float(np.mean(times))
            std = float(np.std(times))
            print('{}: {:.2f} +/- {:.2f} ms'.format(os.path.basename(path), avg, std))

        print('Benchmarking TFLite models (CPU):')
        for name in sorted(tflite_files):
            benchmark_tflite(os.path.join(quant_dir, name))

## Step 14: View Training Logs (Kaggle Working)

In [ ]:
import os
import pandas as pd

results_dir = '/kaggle/working/leaf_models'

if os.path.exists(results_dir):
    csv_files = [f for f in os.listdir(results_dir) if f.endswith('.csv')]
    if csv_files:
        for csv_file in csv_files:
            csv_path = os.path.join(results_dir, csv_file)
            print('\n=== {} ==='.format(csv_file))
            df = pd.read_csv(csv_path)
            print(df.tail(10))
    else:
        print('No CSV files found yet')
else:
    print('Results directory not found yet')

## Step 15: Export Trained Models (Kaggle Files / Save Version)

In [ ]:
import os
import shutil

models_dir = '/kaggle/working/leaf_models'
zip_base = '/kaggle/working/leaf_models'
zip_path = '{}.zip'.format(zip_base)

print('=' * 60)
print('EXPORT TRAINED MODELS')
print('=' * 60)

if os.path.exists(models_dir):
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print('Preparing models archive...')
    shutil.make_archive(zip_base, 'zip', models_dir)
    print('Archive created: {}'.format(zip_path))
    print('Use Save Version to export this zip as a dataset.')
else:
    print('No models found to export')

## Storage Summary

**What happened:**
- Dataset read from Kaggle input (`/kaggle/input/<dataset-name>/processed`)
- Models trained and saved to `/kaggle/working/leaf_models/`
- Quantized TFLite models saved to `/kaggle/working/leaf_models/quantized/`
- Optional export as `leaf_models.zip` in `/kaggle/working`

**Kaggle storage notes:**
- `/kaggle/input` is read-only
- `/kaggle/working` is writable (saved with the notebook run)
- Use Save Version to publish outputs as a Kaggle Dataset

**Next steps:**
1. Save a version to keep the trained models
2. Download `leaf_models.zip` from the notebook output or Kaggle Files
3. Use the INT8 TFLite model in your Flutter app for best on-device speed